In [50]:
# Import librairies

import os
import json
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import io


In [51]:
# Chemins d'accès aux sources

BASE_DIR = os.getcwd()
PATH_CSV = os.path.join(BASE_DIR, "sources", "Chexpert_plus")
PATH_IMAGES = os.path.join(BASE_DIR, "sources", "kaggle")
PATH_JSON = os.path.join(BASE_DIR, "sources", "Chexpert_plus")


In [52]:
# Fonction de chargement CSV et JSON


def chargement_csv_json(fichier_csv, fichier_json):
    
    # Ouverture CSV
    print(f"Chargement du CSV : {fichier_csv}")
    adresse_csv = os.path.join(PATH_CSV, fichier_csv)
    df_csv = pd.read_csv(adresse_csv)
    display(df_csv.head(5))

    # Ouverture JSON
    adresse_json = os.path.join(PATH_JSON, fichier_json)
    print(f"Chargement du JSON : {adresse_json}")
    json_lines_list = []
    with open(adresse_json, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data_chunk = json.loads(line)
                if isinstance(data_chunk, dict):
                    json_lines_list.append(data_chunk)
            except json.JSONDecodeError:
                continue
    # Conversion du json en dataframe
    df_json = pd.DataFrame(json_lines_list)
    display(df_json.head(5))

    # Fusion sur la clé commune 'path_to_image'
    print("Fusion des DataFrames sur la clé 'path_to_image'...")
    df_fusionne = pd.merge(df_csv, df_json, on='path_to_image', how='left', indicator=True)
    df_fusionne["jointure_csv_json"] = df_fusionne["_merge"].map({"both": "OK", "left_only": "PB"})
    df_fusionne = df_fusionne.drop(columns=["_merge"])

    display(df_fusionne.head(5))

    # Affichage stats
    nb_rapport = len(df_csv)
    print(f"Nombre de lignes csv : {nb_rapport}")
    nb_json = len(df_json)
    print(f"Nombre de lignes json : {nb_json}")
    nb_fusion = len(df_fusionne)
    print(f"Nombre de lignes jointure csv + json : {nb_fusion}")
    nb_ok = (df_fusionne["jointure_csv_json"] == "OK").sum()
    print(f"Nombre de lignes avec jointure ok : {nb_ok}")

    # Ouverture image
    def verifier_existence_image(path_to_image, dossier_racine):
        chemin_complet_disque = os.path.normpath(os.path.join(dossier_racine, path_to_image))
        if os.path.exists(chemin_complet_disque):
            return path_to_image 
        else:
            return None

    df_fusionne["chemin_image"] = df_fusionne["path_to_image"].apply(
        lambda x: verifier_existence_image(x,PATH_IMAGES )
        ) 
    
    nb_image_ok = df_fusionne["chemin_image"].notna().sum()
    print(f"Nombre de lignes avec image : {nb_image_ok}")
        
    return df_fusionne
    

In [53]:
FICHIER_CSV = "df_chexpert_plus_240401.csv"
FICHIER_JSON = "impression_fixed.json"

dataset_trav = chargement_csv_json(FICHIER_CSV, FICHIER_JSON)



Chargement du CSV : df_chexpert_plus_240401.csv


,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,section_accession_number,age,sex,race,ethnicity,interpreter_needed,insurance_type,recent_bmi,deceased,split
0,train/patient42142/study5/view1_frontal.jpg,train/patient42142/study5/view1_frontal.dcm,Frontal,AP,patient42142,5,"NARRATIVE:\nChest 1 View, 8-8-2005\n \nHISTORY...","\nChest 1 View, 8-8-2005\n \n",NaN,"61 years Female, ICU patient\n \n",...,\n 9959089\nThis report has been anonymized. A...,62.0,Female,White,Non-Hispanic/Non-Latino,No,Private Insurance,22.2,No,train
1,train/patient42142/study8/view1_frontal.jpg,train/patient42142/study8/view1_frontal.dcm,Frontal,AP,patient42142,8,"NARRATIVE:\nChest 1 View, 7-11-2000\n \nHISTOR...","\nChest 1 View, 7-11-2000\n \n",NaN,"61 years Female, tracheostomy.\n \n",...,\n64048857\nThis report has been anonymized. A...,62.0,Female,White,Non-Hispanic/Non-Latino,No,Private Insurance,22.2,No,train
2,train/patient42142/study2/view1_frontal.jpg,train/patient42142/study2/view1_frontal.dcm,Frontal,AP,patient42142,2,"NARRATIVE:\nChest 1 View, 11-17-2018\n \nHISTO...","\nChest 1 View, 11-17-2018\n \n",NaN,"61 years Female, critical care follow-up\n \n",...,\n#2452\nThis report has been anonymized. All ...,62.0,Female,White,Non-Hispanic/Non-Latino,No,Private Insurance,22.2,No,train
3,train/patient42142/study4/view1_frontal.jpg,train/patient42142/study4/view1_frontal.dcm,Frontal,AP,patient42142,4,NARRATIVE:\nAP PORTABLE UPRIGHT CHEST: May 01 ...,\nAP PORTABLE UPRIGHT CHEST: May 01 at 0518 ho...,ICU protocol. Follow up.\n \n,NaN,...,\n4164064\nThis report has been anonymized. Al...,62.0,Female,White,Non-Hispanic/Non-Latino,No,Private Insurance,22.2,No,train
4,train/patient42142/study3/view1_frontal.jpg,train/patient42142/study3/view1_frontal.dcm,Frontal,AP,patient42142,3,"NARRATIVE:\nEXAM: Chest 1 View, 10/4/2\n \nCLI...","\nEXAM: Chest 1 View, 10/4/2\n \n",61 years Female UPRIGHT PLEASE. ICU PROTOCOL ...,NaN,...,\n49286401\nThis report has been anonymized. A...,62.0,Female,White,Non-Hispanic/Non-Latino,No,Private Insurance,22.2,No,train


Chargement du JSON : c:\Données\FORMATION\fullstack\ProjetFinal\sources\Chexpert_plus\impression_fixed.json


,path_to_image,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding
0,train/patient42142/study5/view1_frontal.jpg,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,1.0,NaN
1,train/patient42142/study8/view1_frontal.jpg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN,NaN,1.0,NaN
2,train/patient42142/study2/view1_frontal.jpg,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN
3,train/patient42142/study4/view1_frontal.jpg,NaN,0.0,NaN,NaN,NaN,-1.0,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN
4,train/patient42142/study3/view1_frontal.jpg,NaN,NaN,NaN,NaN,0.0,0.0,NaN,1.0,NaN,0.0,NaN,NaN,1.0,NaN


Fusion des DataFrames sur la clé 'path_to_image'...


,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json
0,train/patient42142/study5/view1_frontal.jpg,train/patient42142/study5/view1_frontal.dcm,Frontal,AP,patient42142,5,"NARRATIVE:\nChest 1 View, 8-8-2005\n \nHISTORY...","\nChest 1 View, 8-8-2005\n \n",NaN,"61 years Female, ICU patient\n \n",...,NaN,NaN,NaN,0.0,0.0,NaN,NaN,1.0,NaN,OK
1,train/patient42142/study8/view1_frontal.jpg,train/patient42142/study8/view1_frontal.dcm,Frontal,AP,patient42142,8,"NARRATIVE:\nChest 1 View, 7-11-2000\n \nHISTOR...","\nChest 1 View, 7-11-2000\n \n",NaN,"61 years Female, tracheostomy.\n \n",...,NaN,NaN,1.0,0.0,NaN,NaN,NaN,1.0,NaN,OK
2,train/patient42142/study2/view1_frontal.jpg,train/patient42142/study2/view1_frontal.dcm,Frontal,AP,patient42142,2,"NARRATIVE:\nChest 1 View, 11-17-2018\n \nHISTO...","\nChest 1 View, 11-17-2018\n \n",NaN,"61 years Female, critical care follow-up\n \n",...,NaN,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,OK
3,train/patient42142/study4/view1_frontal.jpg,train/patient42142/study4/view1_frontal.dcm,Frontal,AP,patient42142,4,NARRATIVE:\nAP PORTABLE UPRIGHT CHEST: May 01 ...,\nAP PORTABLE UPRIGHT CHEST: May 01 at 0518 ho...,ICU protocol. Follow up.\n \n,NaN,...,-1.0,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,OK
4,train/patient42142/study3/view1_frontal.jpg,train/patient42142/study3/view1_frontal.dcm,Frontal,AP,patient42142,3,"NARRATIVE:\nEXAM: Chest 1 View, 10/4/2\n \nCLI...","\nEXAM: Chest 1 View, 10/4/2\n \n",61 years Female UPRIGHT PLEASE. ICU PROTOCOL ...,NaN,...,0.0,NaN,1.0,NaN,0.0,NaN,NaN,1.0,NaN,OK


Nombre de lignes csv : 223462
Nombre de lignes json : 223462
Nombre de lignes jointure csv + json : 223462
Nombre de lignes avec jointure ok : 223462
Nombre de lignes avec image : 223462


In [54]:
display(dataset_trav.head(5))        

,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json,chemin_image
0,train/patient42142/study5/view1_frontal.jpg,train/patient42142/study5/view1_frontal.dcm,Frontal,AP,patient42142,5,"NARRATIVE:\nChest 1 View, 8-8-2005\n \nHISTORY...","\nChest 1 View, 8-8-2005\n \n",NaN,"61 years Female, ICU patient\n \n",...,NaN,NaN,0.0,0.0,NaN,NaN,1.0,NaN,OK,train/patient42142/study5/view1_frontal.jpg
1,train/patient42142/study8/view1_frontal.jpg,train/patient42142/study8/view1_frontal.dcm,Frontal,AP,patient42142,8,"NARRATIVE:\nChest 1 View, 7-11-2000\n \nHISTOR...","\nChest 1 View, 7-11-2000\n \n",NaN,"61 years Female, tracheostomy.\n \n",...,NaN,1.0,0.0,NaN,NaN,NaN,1.0,NaN,OK,train/patient42142/study8/view1_frontal.jpg
2,train/patient42142/study2/view1_frontal.jpg,train/patient42142/study2/view1_frontal.dcm,Frontal,AP,patient42142,2,"NARRATIVE:\nChest 1 View, 11-17-2018\n \nHISTO...","\nChest 1 View, 11-17-2018\n \n",NaN,"61 years Female, critical care follow-up\n \n",...,NaN,1.0,NaN,1.0,NaN,NaN,1.0,NaN,OK,train/patient42142/study2/view1_frontal.jpg
3,train/patient42142/study4/view1_frontal.jpg,train/patient42142/study4/view1_frontal.dcm,Frontal,AP,patient42142,4,NARRATIVE:\nAP PORTABLE UPRIGHT CHEST: May 01 ...,\nAP PORTABLE UPRIGHT CHEST: May 01 at 0518 ho...,ICU protocol. Follow up.\n \n,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,OK,train/patient42142/study4/view1_frontal.jpg
4,train/patient42142/study3/view1_frontal.jpg,train/patient42142/study3/view1_frontal.dcm,Frontal,AP,patient42142,3,"NARRATIVE:\nEXAM: Chest 1 View, 10/4/2\n \nCLI...","\nEXAM: Chest 1 View, 10/4/2\n \n",61 years Female UPRIGHT PLEASE. ICU PROTOCOL ...,NaN,...,NaN,1.0,NaN,0.0,NaN,NaN,1.0,NaN,OK,train/patient42142/study3/view1_frontal.jpg


E D A

In [55]:
# Extraction du numéro de patient et de l'étude
dataset_trav[["numero_patient", "numero_etude"]] = (
    dataset_trav["path_to_image"].str.extract(r"patient(\d+)/study(\d+)")
)

# Aperçu du résultat
print(dataset_trav[["path_to_image", "numero_patient", "numero_etude"]].head())

                                 path_to_image numero_patient numero_etude
0  train/patient42142/study5/view1_frontal.jpg          42142            5
1  train/patient42142/study8/view1_frontal.jpg          42142            8
2  train/patient42142/study2/view1_frontal.jpg          42142            2
3  train/patient42142/study4/view1_frontal.jpg          42142            4
4  train/patient42142/study3/view1_frontal.jpg          42142            3


In [56]:
# Affichage stat sur chaque colonne

valeurs_renseignees = dataset_trav.count()
valeurs_uniques = dataset_trav.nunique()
valeurs_non_uniques = valeurs_renseignees - valeurs_uniques

df_recap = pd.DataFrame(
    {
        "Valeurs renseignées": valeurs_renseignees,
        "Valeurs uniques": valeurs_uniques,
        "Valeurs non uniques (doublons)": valeurs_non_uniques,
    }
)

df_recap["Valeurs manquantes"] = len(dataset_trav) - valeurs_renseignees

display(df_recap)

,Valeurs renseignées,Valeurs uniques,Valeurs non uniques (doublons),Valeurs manquantes
path_to_image,223462,223462,0,0
path_to_dcm,223462,223462,0,0
frontal_lateral,223462,2,223460,0
ap_pa,191071,4,191067,32391
deid_patient_id,223462,64725,158737,0
patient_report_date_order,223462,92,223370,0
report,223462,223460,2,0
section_narrative,223434,201343,22091,28
section_clinical_history,138866,93944,44922,84596
section_history,38192,26424,11768,185270


On constate qu'il y a 2 rapports strictement identique, nous allons les exclure

In [63]:
dataset_trav.drop_duplicates(subset=['report'], inplace=True)
display (len(dataset_trav))

223460

In [64]:
# Stats nb d'études et d'images par patient
from IPython.display import display

# nombre d'images pour chaque étude de chaque patient
df_images_par_etude = (
    dataset_trav.groupby(["numero_patient", "numero_etude"])
    .size()
    .reset_index(name="nb_image_par_etude")
)

# nombre total d'études uniques pour chaque patient
df_etudes_par_patient = (
    df_images_par_etude.groupby("numero_patient")
    .size()
    .reset_index(name="nb_etude_par_patient")
)

# Fusion deux informations
df_complet = pd.merge(df_images_par_etude, df_etudes_par_patient, on="numero_patient")

# Regroupement par les deux critères pour compter les patients uniques
df_recap_final = (
    df_complet.groupby(["nb_etude_par_patient", "nb_image_par_etude"])
    .agg(Nombre_de_patient=("numero_patient", "nunique"))
    .reset_index()
)

# Réorganisation des colonnes
df_recap_final = df_recap_final[
    ["nb_etude_par_patient", "nb_image_par_etude", "Nombre_de_patient"]
]
df_recap_final.columns = [
    "Nb d'études/patient",
    "Nb d'images/étude",
    "Nombre de patient",
]

# Tri 
df_recap_final = df_recap_final.sort_values(
    ["Nb d'études/patient", "Nb d'images/étude"]
)

display(df_recap_final.style.hide(axis="index"))

Nb d'études/patient,Nb d'images/étude,Nombre de patient
1,1,22767
1,2,10418
1,3,579
2,1,10116
2,2,3826
2,3,292
3,1,5669
3,2,2235
3,3,154
4,1,3581


Pour notre jeu de test pour la faisabilité du projet nous allons sélectionner que les patients ayant une seule étude avec une seule photo  
Puis en sélectionner 100 qui seront nos lignes de référence correctes et générer des cas incorrects

In [72]:
patients_cibles = df_complet[
    (df_complet["nb_etude_par_patient"] == 1)
    & (df_complet["nb_image_par_etude"] == 1)
]
print (len(patients_cibles))

dataset_trav_cible = dataset_trav[
    dataset_trav["numero_patient"].isin(patients_cibles["numero_patient"])
].copy()

print(len(dataset_trav_cible))


22767
22767


In [89]:
# split pour prendre 100 cas corrects, puis constituer 100 cas incorrects (en pointant une mauvaise image sur un rapport 
# quelque soit le profil du patient)

# Prendre 200 lignes au hasard dans le dataset cible
df_trav_200 = dataset_trav_cible.sample(n=300, random_state=42).reset_index(drop=True)

# Splitter les 200 lignes en 3 DataFrames distincts (100 corrects et 2*50 incorrects)
df_trav_rapport_ok = df_trav_200.iloc[0:100].copy()
df_trav_rapport_nok_1 = df_trav_200.iloc[100:150].copy()
df_trav_rapport_nok_2 = df_trav_200.iloc[150:200].copy()

display(df_trav_rapport_nok_1.head(5))

# Création d'un dataframe incorrect en modifiant le chemin de l'image de df_trav_rapport_nok_1 par celles de df_trav_rapport_nok_2
df_trav_rapport_nok_1["chemin_image"] = df_trav_rapport_nok_2["chemin_image"].values
display(df_trav_rapport_nok_1.head(5))

df_trav_rapport_nok_tout_profil = df_trav_rapport_nok_1


,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json,chemin_image,numero_patient,numero_etude,tranche_age
100,train/patient44258/study1/view1_frontal.jpg,train/patient44258/study1/view1_frontal.dcm,Frontal,AP,patient44258,1,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 2-16-201...,\nSINGLE VIEW OF THE CHEST: 2-16-2012 \n \n,"Sixty-five-year-old male, intubated.\n \n",NaN,...,NaN,NaN,1.0,1.0,NaN,OK,train/patient44258/study1/view1_frontal.jpg,44258,1,NaN
101,train/patient59024/study1/view1_frontal.jpg,train/patient59024/study1/view1_frontal.dcm,Frontal,AP,patient59024,1,NARRATIVE:\nChest 1 View: 2-10-2004\n \nHISTOR...,\nChest 1 View: 2-10-2004\n \n,NaN,"Female, 21 years old, postop.\n \n",...,NaN,NaN,NaN,NaN,1.0,OK,train/patient59024/study1/view1_frontal.jpg,59024,1,20.0
102,train/patient19630/study1/view1_frontal.jpg,train/patient19630/study1/view1_frontal.dcm,Frontal,AP,patient19630,1,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\nRADIOGRAPHIC EXAMINATION OF THE CHEST: \n \n,"68 years of age, Female, Rule out pneumothora...",NaN,...,NaN,NaN,NaN,NaN,1.0,OK,train/patient19630/study1/view1_frontal.jpg,19630,1,60.0
103,train/patient58579/study1/view1_frontal.jpg,train/patient58579/study1/view1_frontal.dcm,Frontal,AP,patient58579,1,NARRATIVE:\nCHEST RADIOGRAPH PORTABLE: 11/5/2...,\nCHEST RADIOGRAPH PORTABLE: 11/5/20\n \n,A 65-year-old male post-cardiac surgery.\n \n,NaN,...,NaN,NaN,NaN,NaN,NaN,OK,train/patient58579/study1/view1_frontal.jpg,58579,1,60.0
104,train/patient49449/study1/view1_frontal.jpg,train/patient49449/study1/view1_frontal.dcm,Frontal,AP,patient49449,1,NARRATIVE:\nEXAM: 8-2-11 \n \nCOMPARISON: 8...,\nEXAM: 8-2-11 \n \n,NaN,"58 years Male with Cough, fever\n \n",...,0.0,NaN,NaN,0.0,NaN,OK,train/patient49449/study1/view1_frontal.jpg,49449,1,50.0


,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json,chemin_image,numero_patient,numero_etude,tranche_age
100,train/patient44258/study1/view1_frontal.jpg,train/patient44258/study1/view1_frontal.dcm,Frontal,AP,patient44258,1,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 2-16-201...,\nSINGLE VIEW OF THE CHEST: 2-16-2012 \n \n,"Sixty-five-year-old male, intubated.\n \n",NaN,...,NaN,NaN,1.0,1.0,NaN,OK,train/patient45645/study1/view1_frontal.jpg,44258,1,NaN
101,train/patient59024/study1/view1_frontal.jpg,train/patient59024/study1/view1_frontal.dcm,Frontal,AP,patient59024,1,NARRATIVE:\nChest 1 View: 2-10-2004\n \nHISTOR...,\nChest 1 View: 2-10-2004\n \n,NaN,"Female, 21 years old, postop.\n \n",...,NaN,NaN,NaN,NaN,1.0,OK,train/patient62394/study1/view1_frontal.jpg,59024,1,20.0
102,train/patient19630/study1/view1_frontal.jpg,train/patient19630/study1/view1_frontal.dcm,Frontal,AP,patient19630,1,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\nRADIOGRAPHIC EXAMINATION OF THE CHEST: \n \n,"68 years of age, Female, Rule out pneumothora...",NaN,...,NaN,NaN,NaN,NaN,1.0,OK,train/patient34409/study1/view1_frontal.jpg,19630,1,60.0
103,train/patient58579/study1/view1_frontal.jpg,train/patient58579/study1/view1_frontal.dcm,Frontal,AP,patient58579,1,NARRATIVE:\nCHEST RADIOGRAPH PORTABLE: 11/5/2...,\nCHEST RADIOGRAPH PORTABLE: 11/5/20\n \n,A 65-year-old male post-cardiac surgery.\n \n,NaN,...,NaN,NaN,NaN,NaN,NaN,OK,train/patient58422/study1/view1_frontal.jpg,58579,1,60.0
104,train/patient49449/study1/view1_frontal.jpg,train/patient49449/study1/view1_frontal.dcm,Frontal,AP,patient49449,1,NARRATIVE:\nEXAM: 8-2-11 \n \nCOMPARISON: 8...,\nEXAM: 8-2-11 \n \n,NaN,"58 years Male with Cough, fever\n \n",...,0.0,NaN,NaN,0.0,NaN,OK,train/patient59682/study1/view1_frontal.jpg,49449,1,50.0


In [ ]:
display(len(df_trav_rapport_nok_tout_profil))

50

,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json,chemin_image,numero_patient,numero_etude,tranche_age
100,train/patient44258/study1/view1_frontal.jpg,train/patient44258/study1/view1_frontal.dcm,Frontal,AP,patient44258,1,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 2-16-201...,\nSINGLE VIEW OF THE CHEST: 2-16-2012 \n \n,"Sixty-five-year-old male, intubated.\n \n",NaN,...,NaN,NaN,1.0,1.0,NaN,OK,train/patient45645/study1/view1_frontal.jpg,44258,1,NaN
101,train/patient59024/study1/view1_frontal.jpg,train/patient59024/study1/view1_frontal.dcm,Frontal,AP,patient59024,1,NARRATIVE:\nChest 1 View: 2-10-2004\n \nHISTOR...,\nChest 1 View: 2-10-2004\n \n,NaN,"Female, 21 years old, postop.\n \n",...,NaN,NaN,NaN,NaN,1.0,OK,train/patient62394/study1/view1_frontal.jpg,59024,1,20.0
102,train/patient19630/study1/view1_frontal.jpg,train/patient19630/study1/view1_frontal.dcm,Frontal,AP,patient19630,1,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\nRADIOGRAPHIC EXAMINATION OF THE CHEST: \n \n,"68 years of age, Female, Rule out pneumothora...",NaN,...,NaN,NaN,NaN,NaN,1.0,OK,train/patient34409/study1/view1_frontal.jpg,19630,1,60.0
103,train/patient58579/study1/view1_frontal.jpg,train/patient58579/study1/view1_frontal.dcm,Frontal,AP,patient58579,1,NARRATIVE:\nCHEST RADIOGRAPH PORTABLE: 11/5/2...,\nCHEST RADIOGRAPH PORTABLE: 11/5/20\n \n,A 65-year-old male post-cardiac surgery.\n \n,NaN,...,NaN,NaN,NaN,NaN,NaN,OK,train/patient58422/study1/view1_frontal.jpg,58579,1,60.0
104,train/patient49449/study1/view1_frontal.jpg,train/patient49449/study1/view1_frontal.dcm,Frontal,AP,patient49449,1,NARRATIVE:\nEXAM: 8-2-11 \n \nCOMPARISON: 8...,\nEXAM: 8-2-11 \n \n,NaN,"58 years Male with Cough, fever\n \n",...,0.0,NaN,NaN,0.0,NaN,OK,train/patient59682/study1/view1_frontal.jpg,49449,1,50.0
105,train/patient58849/study1/view1_frontal.jpg,train/patient58849/study1/view1_frontal.dcm,Frontal,AP,patient58849,1,NARRATIVE:\nSINGLE VIEW CHEST RADIOGRAPH: 5/4...,\nSINGLE VIEW CHEST RADIOGRAPH: 5/4/2003. \n...,A 100 year old female with fever and tachypn...,NaN,...,0.0,NaN,NaN,NaN,NaN,OK,train/patient00875/study1/view1_frontal.jpg,58849,1,80.0
106,train/patient49910/study1/view1_frontal.jpg,train/patient49910/study1/view1_frontal.dcm,Frontal,AP,patient49910,1,NARRATIVE:\nPORTABLE CHEST SINGLE VIEW: 7/8/2...,\nPORTABLE CHEST SINGLE VIEW: 7/8/2008 \n,Postoperative. \n,NaN,...,NaN,NaN,NaN,NaN,NaN,OK,train/patient08232/study1/view1_frontal.jpg,49910,1,50.0
107,train/patient64214/study1/view1_frontal.jpg,train/patient64214/study1/view1_frontal.dcm,Frontal,AP,patient64214,1,NARRATIVE:\nCHEST ONE VIEW: july 23\n \n COMP...,\nCHEST ONE VIEW: july 23\n \n,Increased SOB. Rule out effusion or edema.\...,NaN,...,1.0,NaN,NaN,NaN,NaN,OK,train/patient45886/study1/view1_frontal.jpg,64214,1,60.0
108,train/patient64396/study1/view1_frontal.jpg,train/patient64396/study1/view1_frontal.dcm,Frontal,AP,patient64396,1,"NARRATIVE:\nSINGLE VIEW CHEST, 12-12-09:\nCOMP...","\nSINGLE VIEW CHEST, 12-12-09:\n",59 year-old female with A-fib and sick sinus\...,NaN,...,0.0,NaN,NaN,1.0,NaN,OK,train/patient56650/study1/view1_frontal.jpg,64396,1,50.0
109,train/patient62013/study1/view1_frontal.jpg,train/patient62013/study1/view1_frontal.dcm,Frontal,AP,patient62013,1,"NARRATIVE:\nPORTABLE CHEST, SINGLE VIEW: 10/...","\nPORTABLE CHEST, SINGLE VIEW: 10/08\n",Sixty-year-old female with history of breas...,NaN,...,0.0,NaN,NaN,1.0,NaN,OK,train/patient50178/study1/view1_frontal.jpg,62013,1,60.0


In [91]:

# Exclusion des patients déjà traités
patients_a_exclure = set(
    df_trav_rapport_ok["numero_patient"].tolist()
    + df_trav_rapport_nok_1["numero_patient"].tolist()
    + df_trav_rapport_nok_2["numero_patient"].tolist()
)

df_disponible = dataset_trav[
    ~dataset_trav["numero_patient"].isin(patients_a_exclure)
].copy()

dataset_trav["tranche_age"] = (dataset_trav["age"] // 10) * 10
dataset_trav_cible["tranche_age"] = (dataset_trav_cible["age"] // 10) * 10

colonnes_profil = ["sex", "tranche_age", "race", "ethnicity"]

# Compteur taille de chaque groupe démographique dans le dataset cible
cible_counts = dataset_trav_cible.groupby(colonnes_profil).size().reset_index(name="taille_groupe")

# On ne garde que les groupes ayant au moins 5 personnes
profils_valides = cible_counts[cible_counts["taille_groupe"] >= 5][colonnes_profil]

# On filtre dataset_trav_cible pour ne garder que ces profils
df_cible_filtre = pd.merge(dataset_trav_cible, profils_valides, on=colonnes_profil)

# Échantillon de 50 patients et association à un autre chemin d'image 
df_trav_50 = df_disponible.sample(n=50, random_state=42).copy()

donnees_substituees = []

df_cible_dispo = df_cible_filtre.sample(frac=1, random_state=42).copy()

# Pour chaque patient des 50, on cherche son profil similaire
for idx, patient in df_trav_50.iterrows():
    condition = (
        (df_cible_dispo["sex"] == patient["sex"]) &
        (df_cible_dispo["tranche_age"] == patient["tranche_age"]) &
        (df_cible_dispo["race"] == patient["race"]) &
        (df_cible_dispo["ethnicity"] == patient["ethnicity"])
    )
    
    faux_possibles = df_cible_dispo[condition]
    
    if not faux_possibles.empty:
        # On sélectionne le premier donneur trouvé
        index_faux = faux_possibles.index[0]
        faux = faux_possibles.loc[index_faux]
        
        # On retire ce donneur de la liste pour qu'il ne soit plus jamais réutilisé
        df_cible_dispo = df_cible_dispo.drop(index_faux)
        
        donnees_substituees.append({
            "numero_patient": patient["numero_patient"],
            "chemin_image": faux["chemin_image"]
        })
    else:
        # Cas rare : le patient de df_trav_50 a un profil qui n'est pas représenté par au moins 5 personnes dans la cible
        donnees_substituees.append({
            "numero_patient": patient["numero_patient"],
            "chemin_image": None 
        })

df_faux = pd.DataFrame(donnees_substituees)

# On supprime l'ancienne colonne de chemin_image pour y mettre la nouvelle issue du donneur
df_trav_50 = df_trav_50.drop(columns=["chemin_image"])
df_trav_50 = pd.merge(df_trav_50, df_faux, on="numero_patient", how="left")

nb_echecs = df_trav_50["chemin_image"].isna().sum()
print(f"Nombre de patients sans correspondance trouvée : {nb_echecs}")

df_trav_rapport_nok_meme_profil = df_trav_50
display(df_trav_50.head())

Nombre de patients sans correspondance trouvée : 0


,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,Pleural Effusion,Pleural Other,Fracture,Support Devices,No Finding,jointure_csv_json,numero_patient,numero_etude,tranche_age,chemin_image
0,train/patient25867/study2/view2_lateral.jpg,train/patient25867/study2/view2_lateral.dcm,Lateral,NaN,patient25867,4,NARRATIVE:\nTWO VIEWS CHEST: 12-27-21\nCOMPARI...,\nTWO VIEWS CHEST: 12-27-21\n,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,OK,25867,2,50.0,train/patient02114/study1/view1_frontal.jpg
1,train/patient04156/study1/view1_frontal.jpg,train/patient04156/study1/view1_frontal.dcm,Frontal,AP,patient04156,1,NARRATIVE:\nChest 1 View: 12/15/19\n \nHISTORY...,\nChest 1 View: 12/15/19\n \n,NaN,"Male, 22 years old, Trauma.\n \n",...,NaN,NaN,0.0,NaN,1.0,OK,04156,1,20.0,train/patient07921/study1/view1_frontal.jpg
2,train/patient30166/study6/view1_frontal.jpg,train/patient30166/study6/view1_frontal.dcm,Frontal,PA,patient30166,10,NARRATIVE:\nCHEST TWO VIEWS: 12-6-2004\nCOMPAR...,\nCHEST TWO VIEWS: 12-6-2004\n,NaN,of ALL.\n,...,0.0,NaN,NaN,NaN,NaN,OK,30166,6,30.0,train/patient11161/study1/view1_frontal.jpg
3,train/patient37476/study5/view2_frontal.jpg,train/patient37476/study5/view2_frontal.dcm,Frontal,AP,patient37476,5,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\nRADIOGRAPHIC EXAMINATION OF THE CHEST: 1-16-...,"73 years of age, Female, s/p left chest tube ...",NaN,...,NaN,NaN,NaN,0.0,NaN,OK,37476,5,70.0,train/patient05605/study1/view1_frontal.jpg
4,train/patient60136/study5/view2_frontal.jpg,train/patient60136/study5/view2_frontal.dcm,Frontal,AP,patient60136,6,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\nRADIOGRAPHIC EXAMINATION OF THE CHEST: 12/27...,"63 years of age, Male, Pulmonary edema.\n \n",NaN,...,NaN,NaN,NaN,NaN,NaN,OK,60136,5,60.0,train/patient56323/study1/view1_frontal.jpg


In [96]:
# Concaténation dataframe incorrects et suppression des colonnes inutiles

df_trav_rapport_nok = pd.concat([df_trav_rapport_nok_tout_profil, df_trav_rapport_nok_meme_profil], ignore_index=True)

for col in df_trav_rapport_nok.columns:
    display(col)

'path_to_image'

'path_to_dcm'

'frontal_lateral'

'ap_pa'

'deid_patient_id'

...

'jointure_csv_json'

'chemin_image'

'numero_patient'

'numero_etude'

'tranche_age'

In [ ]:
df_trav_rapport_nok = df_trav_rapport_nok.drop("tranche_age", axis=1)

In [114]:
# Vérification que pour le dataframe ok nous avons toujours path_to_image = chemin_image
# et pour le nok path_to_image != chemin_image
ok_valides = (
    df_trav_rapport_ok["path_to_image"] == df_trav_rapport_ok["chemin_image"]
).sum()
ok_total = len(df_trav_rapport_ok)
print (f"Lignes avec valeurs identiques path_to_image et chemin_image pour la dataframe OK : {ok_valides}/{ok_total}")

nok_valides = (
    df_trav_rapport_nok["path_to_image"]
    != df_trav_rapport_nok["chemin_image"]
).sum()
nok_total = len(df_trav_rapport_nok)
print (f"Lignes avec valeurs différentes path_to_image et chemin_image pour la dataframe OK : {nok_valides}/{nok_total}")

structures_identiques = df_trav_rapport_ok.dtypes.equals(df_trav_rapport_ok.dtypes)
print(f"Structure identique entre ok et nok? {structures_identiques}\n")



Lignes avec valeurs identiques path_to_image et chemin_image pour la dataframe OK : 100/100
Lignes avec valeurs différentes path_to_image et chemin_image pour la dataframe OK : 100/100
Structure identique entre ok et nok? True



In [115]:
# Sauvegarde en .csv des dataframes obtenus
from pathlib import Path
output_dir = Path("output")
output_dir.mkdir(
    parents=True, exist_ok=True
)  

df_trav_rapport_ok.to_csv(output_dir / "df_trav_rapport_ok.csv", index=False)
df_trav_rapport_nok.to_csv(output_dir / "df_trav_rapport_nok.csv", index=False)
